## Driver
Used to run the full preprocessing pipeline to generate model-ready `.mat` files.
Run this notebook whenever you add new subjects or want to regenerate the processed data.

#### Generate deep learning data
To generate deep learning data ready data from filtered and normalized time series data.  
Process each raw per-subject `.mat` file (bandpass → resample → z-score).

#### Generate feature data
To generate model-ready feature data in three formats (relies on preprocessed data being generated first):
- **Format 1 — flat_rep**: all windows from one rep concatenated → `(n_reps, 4992)` — for SVM / XGBoost on full gestures
- **Format 2 — flat_window**: each window as an independent sample → `(n_windows, 192)` — for SVM / XGBoost / MLP, real-time capable
- **Format 3 — sequence**: windows stacked in temporal order → `(n_reps, 26, 192)` — for LSTM / GRU / Transformer

---
### Stage 1 — Signal Preprocessing

Processes every raw per-subject `.mat` file: bandpass filter (20–500 Hz) → resample to `TARGET_LENGTH` samples → z-score normalize per channel.  
**Batch** — `batch_process_subjects(input_dir, output_dir, ...)` processes every `.mat` file in `input_dir`.

In [1]:
from preprocessing import batch_process_subjects

APPLY_BANDPASS = True    # Apply 20–500 Hz bandpass filter before resampling
APPLY_ZSCORE   = True    # Apply per-channel z-score normalization after resampling
TARGET_LENGTH  = 1500    # Number of samples after resampling

INPUT_DIR  = '/Users/chrisdollo/Documents/Research/putEMG prime/data/NUG_per_subject'  # Raw per-subject .mat files from the MATLAB pipeline
OUTPUT_DIR = '/Users/chrisdollo/Documents/Research/putEMG prime/data/UG_per_subject'   # Processed per-subject output directory

batch_process_subjects(
    input_dir      = INPUT_DIR,
    output_dir     = OUTPUT_DIR,
    apply_bandpass = APPLY_BANDPASS,
    apply_zscore   = APPLY_ZSCORE,
    target_length  = TARGET_LENGTH,
)

Found 44 subject file(s)
Bandpass: ON | Z-score: ON | Target length: 1500

[1/44] emg_gestures_03_NU_mat.mat
Loading: emg_gestures_03_NU_mat.mat
  Shape: 40 repetitions × 7 gestures

=== Gesture Length Statistics ===
  G1: min=  5124  max= 15373  mean= 10248.2  median= 10248  (n=40)
  G2: min=  5124  max= 15373  mean= 10248.2  median= 10248  (n=40)
  G3: min=  5124  max= 15373  mean= 10248.1  median= 10248  (n=40)
  G6: min=  5124  max= 15373  mean= 10248.3  median= 10248  (n=40)
  G7: min=  5124  max= 15373  mean= 10248.1  median= 10248  (n=40)
  G8: min=  5124  max= 15373  mean= 10248.2  median= 10248  (n=40)
  G9: min=  5124  max= 15373  mean= 10248.1  median= 10248  (n=40)
  Processed 280 repetitions → shape (1500 × 24) each
  Saved → /Users/chrisdollo/Documents/Research/putEMG prime/data/UG_per_subject/emg_gestures_03_U.mat

[2/44] emg_gestures_04_NU_mat.mat
Loading: emg_gestures_04_NU_mat.mat
  Shape: 40 repetitions × 7 gestures

=== Gesture Length Statistics ===
  G1: min=  5124

---
### Stage 2 — Feature Extraction

Extracts 8 hand-crafted features (MAV, RMS, WL, ZC, SSC, VAR, MNF, MDF) × 24 channels = **192 features per window** using a sliding window.  
Run the cells below for whichever format(s) you need. They are independent — run one, some, or all.

| Format | Mode | Output shape | Output file | Use for |
|--------|------|-------------|------------|----------|
| 1 | `flat_rep` | `(n_reps, 4992)` | `.mat` | SVM, XGBoost — full gesture |
| 2 | `flat_window` | `(n_windows, 192)` | `.mat` | SVM, XGBoost, MLP — per window |
| 3 | `sequence` | `(n_reps, 26, 192)` | `.npz` | LSTM, GRU, Transformer |

**Shared config** — set `FEAT_INPUT_DIR` and `FEAT_OUTPUT_DIR` once in the cell below, then run whichever format cells you need.

In [4]:
from feature_extraction import batch_extract_features

WINDOW_SIZE     = 250      # Samples per window (~49 ms at 5120 Hz)
WINDOW_SHIFT    = 50       # Stride between windows (~10 ms)
SAMPLING_RATE   = 5120.0   # Hz — set to effective rate if data was resampled
MODE            = 'sequence'    # 'flat_window', 'flat_rep'
FEAT_INPUT_DIR  = f'/Users/chrisdollo/Documents/Research/putEMG prime/data/UG_per_subject'  # Preprocessed per-subject .mat files (output of Stage 1)
FEAT_OUTPUT_DIR = f'/Users/chrisdollo/Documents/Research/putEMG prime/data/feature_gestures/format_{MODE}'       # Where all feature files will be written

batch_extract_features(
    input_dir     = FEAT_INPUT_DIR,
    output_dir    = FEAT_OUTPUT_DIR,
    mode          = MODE,
    window_size   = WINDOW_SIZE,
    window_shift  = WINDOW_SHIFT,
    sampling_rate = SAMPLING_RATE,
)

Found 44 .mat file(s) in:
  /Users/chrisdollo/Documents/Research/putEMG prime/data/UG_per_subject
Mode  : sequence

─── emg_gestures_03_U.mat  →  features_subject_03_sequence.npz
Loaded  : emg_gestures_03_U.mat
Shape   : 40 repetitions × 7 gestures
Mode    : sequence
Window  : size=250, shift=50, fs=5120.0 Hz
Features: 8 per channel × 24 channels = 192 total

  Gesture G1 (class 0) — accumulated 40 reps so far
  Gesture G2 (class 1) — accumulated 80 reps so far
  Gesture G3 (class 2) — accumulated 120 reps so far
  Gesture G6 (class 3) — accumulated 160 reps so far
  Gesture G7 (class 4) — accumulated 200 reps so far
  Gesture G8 (class 5) — accumulated 240 reps so far
  Gesture G9 (class 6) — accumulated 280 reps so far

Windows per rep : 26
Feature matrix  : X=(280, 26, 192), y=(280,)
Class distribution: {'G1': 40, 'G2': 40, 'G3': 40, 'G6': 40, 'G7': 40, 'G8': 40, 'G9': 40}

Saved   : /Users/chrisdollo/Documents/Research/putEMG prime/data/feature_gestures/format_sequence/features_sub

#### Format 1 — Flat Per Rep (`flat_rep`)

All windows from one gesture rep are concatenated into a single flat vector.  
Output shape: `(n_reps, 4992)` — 26 windows × 192 features, one row per rep.  
Requires all reps to produce the same number of windows (raises an error otherwise).  
Output: `features_subject_<ID>_flat_rep.mat`

#### Format 2 — Flat Per Window (`flat_window`)

Each sliding window becomes one independent sample.  
Output shape: `(n_windows, 192)` — one row per window, label inherited from parent rep.  
Produces the largest dataset (~7280 samples per subject).  
Output: `features_subject_<ID>_flat_window.mat`

#### Format 3 — Sequence Per Rep (`sequence`)

All windows from one gesture rep are stacked in temporal order.  
Output shape: `(n_reps, 26, 192)` — one sequence matrix per rep.  
Requires all reps to produce the same number of windows (raises an error otherwise).  
Output: `features_subject_<ID>_sequence.npz` (NumPy format — 3D arrays don't fit cleanly in `.mat`)